In [ ]:
import pandas as pd
import re 

In [ ]:


def process_and_sort_dataset(file_path):
    # Load the dataset
    df = pd.read_csv(file_path) 
    
    # Remove columns that start with 's_' or 'imp_'
    df_filtered = df[df.columns.drop(list(df.filter(regex='^(s_|imp_)')))]
    
    # Melt the DataFrame to long format
    # We assume 'par_pctile' is already in the dataset as a column
    id_vars = ['par_pctile']  # Keep 'par_pctile' as an identifier in the long format
    df_long = df_filtered.melt(id_vars=id_vars, var_name='category', value_name='outcome_value')
    
    # Extract components (outcome, race, gender) from the 'category' column
    # Updated pattern to include 'pooled' for both race and gender
    pattern = r'(?P<outcome>\w+)_(?P<race>white|black|asian|hisp|natam|other|pooled)_(?P<gender>male|female|pooled)'
    df_long[['outcome', 'race', 'gender']] = df_long['category'].str.extract(pattern)
    
    # Assign outcome_type based on the column name pattern
    def assign_outcome_type(category):
        if category.endswith('_n'):
            return 'n'  # Count of children with non-missing data
        elif category.startswith('count_'):
            return 'count'  # Aggregated count
        else:
            return 'distinct'  # Specific outcome value
    
    df_long['outcome_type'] = df_long['category'].apply(assign_outcome_type)
    
    # Drop the 'category' column as it's now redundant
    df_long.drop(columns=['category'], inplace=True)
    
    outcome_labels = {
        'count': 'Count',
        'kir': 'Individual Income Rank (child earnings)',
        'kir_top01': 'Individual Income Rank - Probability Top 1%',
        'kir_top20': 'Individual Income Rank - Probability Top 20%',
        'kir_24': 'Individual Income Rank (age 24)',
        'kir_26': 'Individual Income Rank (age 26)',
        'kir_29': 'Individual Income Rank (age 29)',
        'hs': 'High School',
        'coll': 'College',
        'comcoll': 'Community College',
        'grad': 'Grad School',
        'has_dad': 'Has Dad',
        'has_mom': 'Has Mom',
        'somecoll': 'Some College',
        'hours_wk': 'Weekly Hours Worked',
        'jail': 'Incarceration',
        'lpov_nbh': 'Low-Poverty Neighborhood',
        'proginc': 'Public Assistance',
        'staytract': 'Stayed in Tract',
        'teenbrth': 'Teen Birth',
        'wgflx_rk': 'Wage Rank',
        'working': 'Working',
        'kfr': 'Mean Household Income',
        'kfr_24': 'Mean Household Income at 24',
        'kfr_26': 'Mean Household Income at 26',
        'kfr_29': 'Mean Household Income at 29',
        'kir_imm': 'Child of Immigrant Individual Income Rank',
        'kir_native': 'Child of US Citizen Individual Income Rank',
        'kir_stycz': 'Individual Income Rank (Same Commuting Zone)',
        'kfr_imm': 'Child of Immigrant Household Income Rank',
        'kfr_native': 'Child of US Citizen Household Income Rank',
        'kfr_stycz': 'Household Income Rank (Same Commuting Zone)',
        'kfr_top01': 'Household Income Rank - Probability Top 1%',
        'kfr_top20': 'Household Income Rank - Probability Top 20%',
        'marr_24': 'Married (24)',
        'marr_26': 'Married (26)', 
        'marr_29': 'Married (29)', 
        'marr_32': 'Married (32)',
        'married': 'Married',
        'pos_hours': 'Positive Working Hours',
        'spouse_rk': 'Spouse Income Rank',
        'staycz': 'Stayed in Childhood Commuting Zone',
        'stayhome': 'Same Address as Parents',
        'staytract': 'Stayed in Childhood Census Tract',
        'two_par': 'Two Parents',
        'work_24': 'Positive Work Hours (24)',
        'work_26': 'Positive Work Hours (26)',
        'work_29': 'Positive Work Hours (29)',
        'work_32': 'Positive Work Hours (32)'

        }
    
    # Unique outcomes
    # ['coll' 'comcoll' 'count' 'grad' 'has_dad' 'has_mom' 'hours_wk' 'hs'
    #  'jail' 'kfr' 'kfr_24' 'kfr_26' 'kfr_29' 'kfr_imm' 'kfr_native'
    #  'kfr_stycz' 'kfr_top01' 'kfr_top20' 'kir' 'kir_24' 'kir_26' 'kir_29'
    #  'kir_imm' 'kir_native' 'kir_stycz' 'kir_top01' 'kir_top20' 'lpov_nbh'
    #  'marr_24' 'marr_26' 'marr_29' 'marr_32' 'married' 'pos_hours' 'proginc'
    #  'somecoll' 'spouse_rk' 'staycz' 'stayhome' 'staytract' 'teenbrth'
    #  'two_par' 'wgflx_rk' 'work_24' 'work_26' 'work_29' 'work_32' 'working']

    df_long['outcome_label'] = df_long['outcome'].map(outcome_labels).fillna(df_long['outcome'])

    # Sort the DataFrame by 'par_pctile', 'race', 'gender', 'outcome', and 'outcome_type'
    df_long.sort_values(by=['par_pctile', 'race', 'gender', 'outcome', 'outcome_label', 'outcome_type'], inplace=True)
    
    # Reorder columns for clarity
    df_long = df_long[['par_pctile', 'race', 'gender', 'outcome', 'outcome_label', 'outcome_type', 'outcome_value']]
    
    # Save the processed DataFrame
    output_file = 'countyoutcomeslong.csv'
    df_long.to_csv(output_file, index=False)
    print(f"Data successfully processed and saved to {output_file}")
    
    return df_long


# Example usage
file_path = 'national_percentile_outcomes.csv'
processed_df = process_and_sort_dataset(file_path)

Data successfully processed and saved to processed_sorted_dataset_with_outcome_type.csv
